# CWRU Data Preparation and Segmentation

This notebook prepares the full Case Western Reserve University (CWRU) vibration dataset for the next stages of our workflow. It performs two main tasks:

**1. Organizing the raw `.mat` files**  
The original CWRU dataset contains mixed signals and irregular naming. Here we extract each drive-end (DE) signal and reorganize everything into a clean and consistent folder structure:

data_CWRU_Organized/  
 ├── 0HP/  
 ├── 1HP/  
 ├── 2HP/  
 ├── 3HP/  
   └── Baseline/ Inner/ Outer/ Ball/  
     *.mat (one DE signal per file)

Each saved file includes the DE signal, the sampling rate (12 kHz), RPM (if available), the class label, the load index, and the original file name.

**2. Segmenting the signals into 1-second windows**  
Every DE signal is divided into overlapping segments using:  
- window length: 12 000 samples (1 s)  
- hop length: 3 000 samples (75% overlap)

For each segment we store the waveform, class label, load label, and file index. After processing all signals, we build the complete dataset and apply a reproducible 70/30 train–test split.

The final output is saved in:

data_CWRU_Organized/data_segments.npz

This file is the standardized input used by all later notebooks in the project: signal processing, image generation, CNN training, and explainability.



In [8]:
from pathlib import Path
import numpy as np
from scipy.io import loadmat, savemat
import math

# Project root = where the notebook is running
PROJECT_ROOT = Path.cwd()

# Folder with original raw data
RAW_ROOT = PROJECT_ROOT / "data_raw_CWRU"

# Folder where the notebook will save the organized dataset
OUT_ROOT = PROJECT_ROOT / "data_CWRU_Organized"

OUT_ROOT.mkdir(exist_ok=True, parents=True)

FS = 12000  # Hz

FOLDER_TO_CLASS = {
    "Baseline":       "Baseline",
    "InnerRaceFault": "Inner",
    "BallFault":      "Ball",
    "OuterRaceFault": "Outer",
}

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------

def infer_load_from_filename(mat_path):
    """
    Extract load index: the number after the final underscore.
    """
    stem = mat_path.stem
    last = stem.split("_")[-1]
    try:
        load = int(last)
    except:
        raise RuntimeError(f"Cannot infer load index from filename: {mat_path.name}")
    if load not in (0,1,2,3):
        raise RuntimeError(f"Unexpected load index {load} in {mat_path.name}")
    return load

def get_class_name(mat_path):
    folder = mat_path.parent.name
    if folder not in FOLDER_TO_CLASS:
        raise RuntimeError(f"Unknown class folder: {folder}")
    return FOLDER_TO_CLASS[folder]

def extract_all_de_signals(mat_path):
    """
    For files with multiple DE signals (e.g., Normal_2.mat),
    return a list of:
      {"id": "X097", "signal": array, "rpm": float}
    """
    data = loadmat(mat_path)
    keys = [k for k in data.keys() if not k.startswith("__")]

    de_keys = [k for k in keys if k.endswith("_DE_time")]
    if len(de_keys) == 0:
        raise RuntimeError(f"No DE signal found in {mat_path.name}")

    results = []

    for de_key in de_keys:
        id_prefix = de_key.replace("_DE_time", "")
        signal = np.squeeze(data[de_key]).astype(np.float32)

        rpm_keys = [k for k in keys if k.startswith(id_prefix) and k.endswith("RPM")]
        if len(rpm_keys) == 1:
            rpm = float(np.squeeze(data[rpm_keys[0]]))
        else:
            rpm = -1.0  # baseline case

        results.append({
            "id": id_prefix,
            "signal": signal,
            "rpm": rpm,
        })

    return results

# ---------------------------------------------------------
# MAIN ORGANIZATION
# ---------------------------------------------------------

def reorganize_cwru():
    OUT_ROOT.mkdir(exist_ok=True)
    counts = {}

    for folder_name, class_name in FOLDER_TO_CLASS.items():
        src_folder = RAW_ROOT / folder_name

        for mat_path in sorted(src_folder.glob("*.mat")):

            load_idx = infer_load_from_filename(mat_path)
            cls = get_class_name(mat_path)

            parts = extract_all_de_signals(mat_path)

            for p in parts:
                sig = p["signal"]
                rpm = p["rpm"]
                prefix = mat_path.stem   # e.g. "IR007_0" or "Normal_2"

                # build output folder
                out_folder = OUT_ROOT / f"{load_idx}HP" / cls
                out_folder.mkdir(parents=True, exist_ok=True)

                # final file name rule
                if len(parts) == 1:
                    out_name = f"{prefix}_DEonly.mat"
                else:
                    out_name = f"{prefix}_{p['id']}_DEonly.mat"

                # save
                savemat(out_folder / out_name, {
                    "signal": sig,
                    "fs": FS,
                    "rpm": rpm,
                    "original_file": mat_path.name,
                    "id": p["id"],
                    "load": load_idx,
                    "class_name": cls,
                })

                key = (load_idx, cls)
                counts[key] = counts.get(key, 0) + 1

    print("\n=== Summary ===")
    for load in sorted({k[0] for k in counts}):
        for cls in ["Baseline", "Inner", "Ball", "Outer"]:
            print(f"Load {load}HP  {cls:8s}: {counts.get((load, cls), 0)} files")


if __name__ == "__main__":
    reorganize_cwru()



=== Summary ===
Load 0HP  Baseline: 1 files
Load 0HP  Inner   : 4 files
Load 0HP  Ball    : 4 files
Load 0HP  Outer   : 7 files
Load 1HP  Baseline: 1 files
Load 1HP  Inner   : 4 files
Load 1HP  Ball    : 4 files
Load 1HP  Outer   : 7 files
Load 2HP  Baseline: 2 files
Load 2HP  Inner   : 4 files
Load 2HP  Ball    : 4 files
Load 2HP  Outer   : 7 files
Load 3HP  Baseline: 1 files
Load 3HP  Inner   : 4 files
Load 3HP  Ball    : 4 files
Load 3HP  Outer   : 7 files


In [9]:
# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------

ORG_ROOT = PROJECT_ROOT / "data_CWRU_Organized"

FS = 12000            # Hz
WIN_LEN = 12000       # 1 second
HOP_LEN = 3000        # stride

# Class mapping to match Chen's order: (normal, inner, outer, ball)
CLASS_NAME_TO_ID = {
    "Baseline": 0,   # normal
    "Inner":    1,
    "Outer":    2,
    "Ball":     3,
}

RNG_SEED = 2020       # for reproducible split

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------

def segment_1d_signal(sig, win_len, hop_len):
    sig = np.asarray(sig).flatten()
    n = len(sig)
    if n < win_len:
        return np.empty((0, win_len), dtype=np.float32)

    starts = np.arange(0, n - win_len + 1, hop_len)
    segs = np.stack([sig[s:s+win_len] for s in starts], axis=0)
    return segs.astype(np.float32)

# ---------------------------------------------------------
# BUILD FULL SEGMENT DATASET
# ---------------------------------------------------------

def build_segments_from_organized():
    all_X = []
    all_y_class = []
    all_y_load = []
    all_file_idx = []
    file_names = []

    file_counter = 0

    # Loop loads: 0HP, 1HP, 2HP, 3HP
    for load_folder in sorted(ORG_ROOT.glob("*HP")):
        load_name = load_folder.name   # e.g. "0HP"
        load_idx = int(load_name[0])   # 0,1,2,3

        for class_name in ["Baseline", "Inner", "Outer", "Ball"]:
            cls_folder = load_folder / class_name
            if not cls_folder.is_dir():
                continue

            cls_id = CLASS_NAME_TO_ID[class_name]

            for mat_path in sorted(cls_folder.glob("*.mat")):
                data = loadmat(mat_path)
                sig = np.squeeze(data["signal"]).astype(np.float32)

                # Safety check on fs (should be 12000)
                if "fs" in data:
                    fs_here = int(np.squeeze(data["fs"]))
                    if fs_here != FS:
                        print(f"Warning: fs={fs_here} in {mat_path.name}, expected {FS}")

                # Segment
                segs = segment_1d_signal(sig, WIN_LEN, HOP_LEN)
                if segs.shape[0] == 0:
                    continue

                n_seg = segs.shape[0]
                all_X.append(segs)
                all_y_class.append(np.full(n_seg, cls_id, dtype=np.int64))
                all_y_load.append(np.full(n_seg, load_idx, dtype=np.int64))
                all_file_idx.append(np.full(n_seg, file_counter, dtype=np.int64))
                file_names.append(mat_path.name)

                file_counter += 1

    X = np.vstack(all_X)
    y_class = np.concatenate(all_y_class)
    y_load = np.concatenate(all_y_load)
    file_idx = np.concatenate(all_file_idx)
    file_names_arr = np.array(file_names, dtype=object)

    return X, y_class, y_load, file_idx, file_names_arr

# ---------------------------------------------------------
# TRAIN / TEST SPLIT (70 / 30)
# ---------------------------------------------------------

def train_test_split_segments(X, y_class, y_load, file_idx, file_names,
                              train_ratio=0.7, seed=RNG_SEED):

    N = X.shape[0]
    rng = np.random.RandomState(seed)
    perm = rng.permutation(N)

    X = X[perm]
    y_class = y_class[perm]
    y_load = y_load[perm]
    file_idx = file_idx[perm]

    n_train = int(math.floor(train_ratio * N))

    split = {
        "X_train": X[:n_train],
        "y_class_train": y_class[:n_train],
        "y_load_train": y_load[:n_train],
        "file_idx_train": file_idx[:n_train],

        "X_test": X[n_train:],
        "y_class_test": y_class[n_train:],
        "y_load_test": y_load[n_train:],
        "file_idx_test": file_idx[n_train:],

        "file_names": file_names,
        "perm": perm,          # to reproduce the exact split later if needed
    }

    return split

# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------

if __name__ == "__main__":
    X, y_class, y_load, file_idx, file_names = build_segments_from_organized()
    print("Total segments:", X.shape[0])
    print("Class counts:", {c:int((y_class==c).sum()) for c in np.unique(y_class)})
    print("Load counts:", {l:int((y_load==l).sum()) for l in np.unique(y_load)})

    split = train_test_split_segments(X, y_class, y_load, file_idx, file_names, train_ratio=0.7)

    # Save everything in one npz file
    out_path = ORG_ROOT / "data_segments.npz"
    np.savez(out_path, **split)

    print(f"Saved dataset to: {out_path}")
    print(f"Train segments: {split['X_train'].shape[0]}")
    print(f"Test segments : {split['X_test'].shape[0]}")


Total segments: 2930
Class counts: {np.int64(0): 710, np.int64(1): 592, np.int64(2): 1036, np.int64(3): 592}
Load counts: {np.int64(0): 633, np.int64(1): 713, np.int64(2): 871, np.int64(3): 713}
Saved dataset to: C:\Users\Nicolas\XAI\COE691_XAI_Project-Copy1\data_CWRU_Organized\data_segments.npz
Train segments: 2051
Test segments : 879
